# Mission Control — laboratório eBPF (libbpf) + Docker

No **host** basta ter Docker em Linux com BTF (`/sys/kernel/btf/vmlinux`). **Não** é necessário instalar `clang`, `llvm` nem `bpftool` no host: a compilação e o `./loader` rodam dentro do container **`ebpf_loader`** (privilegiado, rede e PID do host, volumes para BTF, bpf e debug).

Abra este notebook com o diretório de trabalho na pasta **`mission-control-lab`** (onde estão `Makefile`, `docker-compose.yml`, `Dockerfile.loader` e os `.c`).

### 1. Subir apps em deadlock + container do loader

`docker compose up -d` sobe `mc_app_a`, `mc_app_b` na rede `mission_control_net` e o `ebpf_loader` em segundo plano (`tail -f /dev/null`).

In [ ]:
!docker compose up -d --build

### 2. Compilar eBPF e o loader **dentro** do `ebpf_loader`

O diretório do projeto está montado em `/opt/mission`; os artefatos (`vmlinux.h`, `.o`, `.skel.h`, binário `loader`) aparecem também no host.

In [ ]:
!docker exec ebpf_loader sh -c 'cd /opt/mission && make clean 2>/dev/null; make all'

### 3. Estado dos containers e da rede Docker

In [ ]:
!docker ps -a --filter "name=mc_app" ; docker ps -a --filter "name=ebpf_loader" ; echo "---" && docker network inspect mission_control_net --format '{{json .}}' | head -c 2000

### 4. Rodar o loader no container (TC na `br-…` + ringbuf)

O `ebpf_loader` usa `network_mode: host`, então vê a interface `br-` da rede `mission_control_net`. Esta célula **bloqueia** até você interromper a execução do notebook (Stop) ou o processo terminar; em outro terminal pode acompanhar `docker logs mc_app_a` / `mc_app_b`.

**Sem `sudo` no host** — o processo já roda como root dentro do container privilegiado.

In [ ]:
!BR=$(docker network inspect mission_control_net -f '{{.Id}}' | head -c 12) && echo "Bridge: br-$BR" && docker exec ebpf_loader sh -c "cd /opt/mission && ./loader br-$BR"

### 5. Logs dos apps (RST / saída do `recv`)

Após ~5 s sem payload TCP nas conexões ativas, o kernel deve registrar o alerta e o RST; os processos saem do `recv()`. A célula abaixo mostra somente os logs da execução atual dos containers e renderiza o incidente em um painel visual.

In [ ]:
import html
import re
import subprocess
from IPython.display import HTML, display

ANSI = re.compile(r'\x1b\[[0-?]*[ -/]*[@-~]')

def container_started_at(container):
    return subprocess.check_output(
        ['docker', 'inspect', '-f', '{{.State.StartedAt}}', container],
        text=True,
        stderr=subprocess.STDOUT,
    ).strip()

def current_logs(container):
    since = container_started_at(container)
    result = subprocess.run(
        ['docker', 'logs', '--since', since, '--tail', '80', container],
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        check=False,
    )
    lines = [line for line in ANSI.sub('', result.stdout).splitlines() if line.strip()]
    for index, line in enumerate(lines):
        if 'FINALIZANDO PROCESSO COM SEGURANÇA' in line:
            end = index + 2 if index + 1 < len(lines) and '╚' in lines[index + 1] else index + 1
            return lines[:end]
    return lines

def classify(line):
    if 'RST RECEBIDO' in line or 'INCIDENTE CRÍTICO' in line or line.startswith('╔') or line.startswith('╚'):
        return 'danger'
    if 'RCV RETORNANDO' in line:
        return 'warn'
    if 'DEADLOCK TRATADO' in line:
        return 'ok'
    if 'FINALIZANDO' in line:
        return 'stop'
    if 'recv() bloqueante' in line:
        return 'wait'
    if 'ESTABLISHED' in line or 'Canal' in line:
        return 'link'
    return 'line'

def render_line(line):
    return f'<div class="log {classify(line)}">{html.escape(line)}</div>'

def panel(title, container, accent):
    lines = current_logs(container)
    status = 'INCIDENTE TRATADO' if any('DEADLOCK TRATADO' in line for line in lines) else 'AGUARDANDO RST'
    body = ''.join(render_line(line) for line in lines) or '<div class="log wait">Sem logs nesta execução.</div>'
    return f'''
    <section class="mc-panel" style="--accent:{accent}">
      <div class="mc-panel-title">
        <span>{title}</span>
        <strong>{status}</strong>
      </div>
      <div class="mc-logbox">{body}</div>
    </section>
    '''

style = '''
<style>
.mc-dashboard { font-family: ui-monospace, SFMono-Regular, Menlo, Consolas, monospace; background:#071018; color:#d9f7ff; padding:18px; border-radius:10px; border:1px solid #21475a; }
.mc-title { color:#7dd3fc; font-size:20px; font-weight:800; letter-spacing:.04em; margin-bottom:4px; }
.mc-subtitle { color:#8aa7b4; margin-bottom:14px; }
.mc-grid { display:grid; grid-template-columns:repeat(auto-fit, minmax(360px, 1fr)); gap:14px; }
.mc-panel { border:1px solid var(--accent); border-left:5px solid var(--accent); border-radius:8px; background:#0b1720; overflow:hidden; }
.mc-panel-title { display:flex; justify-content:space-between; gap:12px; padding:10px 12px; color:white; background:#0f2230; }
.mc-panel-title span { font-weight:800; color:var(--accent); }
.mc-panel-title strong { color:#86efac; }
.mc-logbox { padding:10px 12px; }
.log { padding:4px 0; white-space:pre-wrap; border-bottom:1px solid rgba(255,255,255,.05); }
.danger { color:#fecaca; font-weight:800; background:rgba(239,68,68,.12); }
.warn { color:#fde68a; font-weight:800; }
.ok { color:#86efac; font-weight:800; }
.stop { color:#67e8f9; font-weight:800; }
.wait { color:#fbbf24; }
.link { color:#93c5fd; }
.line { color:#cbd5e1; }
</style>
'''

display(HTML(style + f'''
<div class="mc-dashboard">
  <div class="mc-title">MISSION CONTROL | LOGS DOS APPS</div>
  <div class="mc-subtitle">Logs isolados por container desde o último start. Sem histórico antigo misturado.</div>
  <div class="mc-grid">
    {panel('APP A · 172.28.0.2:8081', 'mc_app_a', '#22d3ee')}
    {panel('APP B · 172.28.0.3:8082', 'mc_app_b', '#a78bfa')}
  </div>
</div>
'''))

### Encerrar (opcional)

In [ ]:
!docker compose down